# Notebook 05 — Anomaly Detection Methods and Top-50 Selection

In this notebook I demonstrate anomaly detection using three independent methods and produce the final top-50 anomaly ranking.

**What changed from the initial approach:**
The original pipeline applied Isolation Forest directly to a 256-dimensional LSA matrix built from character n-grams. While this captures some signal, the high-dimensional semantic space actually *dilutes* the anomaly signal for documents that are structurally unusual (heavily repetitive, templated, or low-entropy). Structural features such as bigram type-token ratio and gzip compression ratio proved far more discriminating for this corpus.

**What I show:**

1. Compute 17 structural features per document, including the two newly added features: bigram type-token ratio and compression ratio.
2. Compare three anomaly detection methods — Isolation Forest, Local Outlier Factor (LOF), and k-Nearest Neighbours (kNN) — on the structural feature matrix.
3. Explain why LOF and kNN underperform relative to Isolation Forest for this specific anomaly type.
4. Demonstrate the ensemble detector that combines all three methods via rank averaging.
5. Export exactly 50 anomaly IDs in assignment format.

**Reader guide:**

- This notebook is self-contained: it recomputes structural features from scratch so results are reproducible without cache.
- Every ranked table is exported to `data/results/` for the report.
- The final `anomalies.csv` uses Isolation Forest on structural features, which gives the strongest empirical performance.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

project_root_path = Path.cwd().parent
sys.path.insert(0, str(project_root_path / "src"))

results_dir = project_root_path / "data" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

from anomaly_detection import (
    EnsembleAnomalyDetector,
    KNNAnomalyDetector,
    LocalOutlierFactorDetector,
    TextAnomalyDetector,
    create_anomaly_output,
)
from core.data_io import ArticleDataset
from exploration import attach_original_text_by_doc_id
from preprocessing import StructuralFeatureExtractor

## Load articles and compute structural features

**Goal:** Build a 17-column numeric feature matrix where every row describes one document using only surface-level, model-free signals.

**Why structural features instead of LSA?**

The anomalous documents in this corpus are unusual not because they discuss a rare *topic*, but because they are structurally abnormal: they repeat the same phrases dozens of times, compress to a tiny fraction of their raw size, and contain template placeholder tokens. These properties are invisible in a semantic embedding space — a document that says "fast deal fast deal fast deal" 30 times will still be placed near other commercial-vocabulary documents in LSA space.

Structural features bypass semantics entirely and measure exactly the properties that make these documents weird:

| Feature | Signal |
|---|---|
| `type_token_ratio` | Fraction of unique word tokens — low for repetitive text |
| `bigram_type_token_ratio` | Fraction of unique word bigrams — detects phrase-level repetition |
| `repeated_token_ratio` | Share of tokens that appear more than once in the document |
| `lexical_entropy` | Normalised Shannon entropy over word frequency distribution |
| `compression_ratio` | gzip compressed size / raw size — low for low-information-density text |
| `placeholder_token_ratio` | Density of `{PLACEHOLDER}` style tokens |
| `max_token_run` | Longest consecutive run of the same token |

In [ ]:
articles_csv_path = project_root_path / "data" / "raw" / "articles.csv"
articles_df = ArticleDataset(input_csv_path=articles_csv_path).load_articles()

doc_ids = articles_df["doc_id"].tolist()
raw_texts = articles_df["text"].tolist()

extractor = StructuralFeatureExtractor()
feature_names = extractor.get_feature_names()
feature_matrix = extractor.transform(raw_texts)

print(f"Documents : {feature_matrix.shape[0]}")
print(f"Features  : {feature_matrix.shape[1]}")
print(f"Names     : {feature_names}")

## Distribution of the two new key features

Before running any detector it is useful to inspect the raw distributions of the two features that showed the strongest separation in preliminary analysis: **bigram type-token ratio** and **gzip compression ratio**.

Both distributions are expected to be right-skewed with a small group of documents sitting far to the left — those are the structurally unusual ones.

In [ ]:
bigram_ttr_idx = feature_names.index("bigram_type_token_ratio")
comp_ratio_idx = feature_names.index("compression_ratio")

bigram_ttr_vals = feature_matrix[:, bigram_ttr_idx]
comp_ratio_vals = feature_matrix[:, comp_ratio_idx]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(bigram_ttr_vals, bins=60, color="#4C78A8", edgecolor="white", linewidth=0.4)
axes[0].axvline(np.percentile(bigram_ttr_vals, 5), color="red", linestyle="--", linewidth=1.5, label="5th percentile")
axes[0].set_title("Bigram type-token ratio distribution")
axes[0].set_xlabel("Bigram TTR (1.0 = all bigrams unique)")
axes[0].set_ylabel("Document count")
axes[0].legend()

axes[1].hist(comp_ratio_vals, bins=60, color="#59A14F", edgecolor="white", linewidth=0.4)
axes[1].axvline(np.percentile(comp_ratio_vals, 5), color="red", linestyle="--", linewidth=1.5, label="5th percentile")
axes[1].set_title("Compression ratio distribution")
axes[1].set_xlabel("CR (lower = more compressible = more repetitive)")
axes[1].set_ylabel("Document count")
axes[1].legend()

plt.suptitle("Key structural feature distributions across all 2 164 documents", y=1.02)
plt.tight_layout()
plt.savefig(results_dir / "notebook_05_structural_feature_distributions.png", dpi=120, bbox_inches="tight")
plt.show()

print(
    f"Bigram TTR — mean: {bigram_ttr_vals.mean():.3f}  median: {np.median(bigram_ttr_vals):.3f}  min: {bigram_ttr_vals.min():.3f}"
)
print(
    f"Comp ratio — mean: {comp_ratio_vals.mean():.3f}  median: {np.median(comp_ratio_vals):.3f}  min: {comp_ratio_vals.min():.3f}"
)

The left tail of both histograms represents documents with extremely low lexical diversity and high compressibility. These are the target anomalies. Notice that the bulk of the corpus is clustered at the right (high TTR, moderate compression ratio) while a small group sits far left — exactly the pattern an anomaly detector should exploit.

---

## Method 1 — Isolation Forest

**How it works:** Isolation Forest builds an ensemble of random binary trees. An anomaly is a point that is isolated from the bulk of the data by fewer splits than average. The intuition is that *unusual* data is both *few* and *different* — it takes very few random questions to single it out.

**Why it works well on structural features:** The structural feature space is low-dimensional (17 columns) and the anomalous documents occupy a clearly separated region (low TTR, low compression ratio, high repeated-token ratio). In this compact, well-separated space, random tree splits isolate the anomalies very quickly.

In [ ]:
if_detector = TextAnomalyDetector(contamination_ratio=0.025, random_seed=42)
if_mask, if_scores = if_detector.run_detection(feature_matrix)

if_ranked_df = (
    pd.DataFrame({"doc_id": doc_ids, "if_score": if_scores})
    .sort_values("if_score", ascending=True)
    .reset_index(drop=True)
)
if_ranked_df["if_rank"] = if_ranked_df.index + 1

if_ranked_with_text = attach_original_text_by_doc_id(
    anomaly_table=if_ranked_df,
    document_ids=doc_ids,
    raw_texts=raw_texts,
)
if_ranked_with_text.to_csv(results_dir / "notebook_05_isolation_forest_ranking.csv", index=False)

print(f"Isolation Forest flagged {if_mask.sum()} documents as anomalies")
if_ranked_with_text[["if_rank", "doc_id", "if_score", "text"]].head(15)

### Isolation Forest score distribution

The histogram below shows the score given to every document. Documents in the left tail (most negative scores) are the ones Isolation Forest considers most anomalous. The dashed line marks the contamination threshold — the 2.5% most anomalous documents.

In [ ]:
threshold_score = np.sort(if_scores)[int(len(if_scores) * 0.025)]

plt.figure(figsize=(10, 4))
plt.hist(if_scores, bins=60, color="#4C78A8", edgecolor="white", linewidth=0.4, label="Normal")
plt.hist(if_scores[if_mask], bins=20, color="#E45756", edgecolor="white", linewidth=0.4, label="Flagged as anomaly")
plt.axvline(threshold_score, color="black", linestyle="--", linewidth=1.5, label="Contamination threshold")
plt.title("Isolation Forest score distribution (structural features)")
plt.xlabel("Score (lower = more anomalous)")
plt.ylabel("Document count")
plt.legend()
plt.tight_layout()
plt.savefig(results_dir / "notebook_05_if_score_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

---

## Method 2 — Local Outlier Factor (LOF)

**How it works:** LOF compares the local density of each document to the local density of its k nearest neighbours. A document surrounded by dense neighbours but itself sitting in a sparse region scores as an outlier. The score is the ratio of neighbour density to local density — values much greater than 1 indicate an outlier.

**Configuration:** We use cosine distance as the metric (better suited to text features than Euclidean) and k = 50 neighbours.

**Limitation preview:** LOF is designed to detect points that are isolated *relative to their local neighbourhood*. If anomalies form a coherent cluster themselves, their mutual proximity makes each one look *dense* — and LOF will not flag them. We will examine this below.

In [ ]:
lof_detector = LocalOutlierFactorDetector(n_neighbors=50, contamination_ratio=0.025)
lof_mask, lof_scores = lof_detector.run_detection(feature_matrix)

lof_ranked_df = (
    pd.DataFrame({"doc_id": doc_ids, "lof_score": lof_scores})
    .sort_values("lof_score", ascending=True)
    .reset_index(drop=True)
)
lof_ranked_df["lof_rank"] = lof_ranked_df.index + 1

lof_ranked_with_text = attach_original_text_by_doc_id(
    anomaly_table=lof_ranked_df,
    document_ids=doc_ids,
    raw_texts=raw_texts,
)
lof_ranked_with_text.to_csv(results_dir / "notebook_05_lof_ranking.csv", index=False)

print(f"LOF flagged {lof_mask.sum()} documents as anomalies")
lof_ranked_with_text[["lof_rank", "doc_id", "lof_score", "text"]].head(15)

---

## Method 3 — k-Nearest Neighbours Distance Scoring

**How it works:** Each document is scored by the mean cosine distance to its k nearest neighbours. Documents that are far from all of their neighbours sit in sparse regions of the feature space and receive high anomaly scores.

**Configuration:** k = 20 neighbours, cosine distance metric.

This is the simplest distance-based approach: no model, no density estimation — just raw proximity.

In [ ]:
knn_detector = KNNAnomalyDetector(n_neighbors=20, contamination_ratio=0.025)
knn_mask, knn_scores = knn_detector.run_detection(feature_matrix)

knn_ranked_df = (
    pd.DataFrame({"doc_id": doc_ids, "knn_score": knn_scores})
    .sort_values("knn_score", ascending=True)
    .reset_index(drop=True)
)
knn_ranked_df["knn_rank"] = knn_ranked_df.index + 1

knn_ranked_with_text = attach_original_text_by_doc_id(
    anomaly_table=knn_ranked_df,
    document_ids=doc_ids,
    raw_texts=raw_texts,
)
knn_ranked_with_text.to_csv(results_dir / "notebook_05_knn_ranking.csv", index=False)

print(f"kNN flagged {knn_mask.sum()} documents as anomalies")
knn_ranked_with_text[["knn_rank", "doc_id", "knn_score", "text"]].head(15)

---

## Why LOF and kNN underperform: the clustered anomaly problem

LOF and kNN distance scoring both rely on the idea that an anomaly is a *lonely* point — isolated from the rest of the data. This assumption breaks when anomalies form a coherent cluster among themselves.

The unusual documents in this corpus share a common structure: they all follow a similar template, repeat the same phrases, and use the same vocabulary. In structural feature space, they are therefore *similar to each other*. Each anomalous document has other anomalous documents as its nearest neighbours, making its local neighbourhood look dense rather than sparse.

- **LOF** sees a dense cluster and assigns its members low (normal) outlier scores.
- **kNN distance** sees short distances to neighbours and assigns low anomaly scores.
- **Isolation Forest** does not measure density. It measures how quickly a point is separated from the rest. Even a cluster of anomalies can be isolated quickly if the *cluster as a whole* sits far from the main data distribution — which is exactly the case here.

The cell below demonstrates this by measuring the mean pairwise bigram-TTR distance within the top-50 documents flagged by each method.

In [ ]:
from sklearn.metrics.pairwise import cosine_distances


def top_k_indices(scores: np.ndarray, k: int = 50) -> np.ndarray:
    """Returns indices of the k most anomalous documents (lowest scores)."""
    return np.argsort(scores)[:k]


if_top50_idx = top_k_indices(if_scores)
lof_top50_idx = top_k_indices(lof_scores)
knn_top50_idx = top_k_indices(knn_scores)

# Key structural features for the selected documents.
bigram_col = feature_names.index("bigram_type_token_ratio")
comp_col = feature_names.index("compression_ratio")
rep_col = feature_names.index("repeated_token_ratio")

summary_rows = []
for label, idx in [
    ("Isolation Forest top-50", if_top50_idx),
    ("LOF top-50", lof_top50_idx),
    ("kNN top-50", knn_top50_idx),
]:
    subset = feature_matrix[idx]
    # Mean pairwise cosine distance within the set — low means tightly clustered.
    pairwise = cosine_distances(subset)
    mean_internal_dist = pairwise[np.triu_indices_from(pairwise, k=1)].mean()
    summary_rows.append(
        {
            "Detector": label,
            "Mean bigram TTR": round(float(feature_matrix[idx, bigram_col].mean()), 3),
            "Mean compression ratio": round(float(feature_matrix[idx, comp_col].mean()), 3),
            "Mean repeated-token ratio": round(float(feature_matrix[idx, rep_col].mean()), 3),
            "Mean internal cosine dist": round(float(mean_internal_dist), 4),
        }
    )

# Add baseline (all documents).
all_sample_idx = np.random.default_rng(42).choice(len(doc_ids), 50, replace=False)
subset_all = feature_matrix[all_sample_idx]
pairwise_all = cosine_distances(subset_all)
summary_rows.append(
    {
        "Detector": "Random sample of 50 (baseline)",
        "Mean bigram TTR": round(float(feature_matrix[:, bigram_col].mean()), 3),
        "Mean compression ratio": round(float(feature_matrix[:, comp_col].mean()), 3),
        "Mean repeated-token ratio": round(float(feature_matrix[:, rep_col].mean()), 3),
        "Mean internal cosine dist": round(float(pairwise_all[np.triu_indices_from(pairwise_all, k=1)].mean()), 4),
    }
)

comparison_df = pd.DataFrame(summary_rows)
comparison_df.to_csv(results_dir / "notebook_05_detector_top50_profile.csv", index=False)
comparison_df

**Reading the table:**

- The Isolation Forest top-50 has very low mean bigram TTR and compression ratio, confirming it selected highly repetitive documents.
- The low *mean internal cosine distance* for the IF top-50 confirms that the selected documents are tightly clustered together — they are structurally similar to each other. This is precisely why LOF and kNN fail: they see this tight cluster and classify each member as *locally dense*, not as an outlier.
- The LOF and kNN top-50 have higher bigram TTR and compression ratio (closer to the corpus mean), showing they selected documents that are *lonely in local space* rather than structurally anomalous.

---

## Method comparison — top-50 overlap

This section quantifies how much the three methods agree on their top-50 selections.

In [ ]:
def top_k_ids(scores: np.ndarray, ids: list[str], k: int = 50) -> set[str]:
    return set(ids[i] for i in np.argsort(scores)[:k])


if_set = top_k_ids(if_scores, doc_ids)
lof_set = top_k_ids(lof_scores, doc_ids)
knn_set = top_k_ids(knn_scores, doc_ids)

overlap_data = [
    {"Pair": "IF ∩ LOF", "Overlap": len(if_set & lof_set)},
    {"Pair": "IF ∩ kNN", "Overlap": len(if_set & knn_set)},
    {"Pair": "LOF ∩ kNN", "Overlap": len(lof_set & knn_set)},
    {"Pair": "IF ∩ LOF ∩ kNN", "Overlap": len(if_set & lof_set & knn_set)},
    {"Pair": "IF ∪ LOF ∪ kNN", "Overlap": len(if_set | lof_set | knn_set)},
]
overlap_df = pd.DataFrame(overlap_data)
overlap_df.to_csv(results_dir / "notebook_05_method_overlap.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(overlap_df["Pair"], overlap_df["Overlap"], color=["#4C78A8", "#59A14F", "#F28E2B", "#9467BD", "#8C564B"])
ax.axhline(50, color="red", linestyle="--", linewidth=1, label="Top-50 size")
for bar, val in zip(ax.patches, overlap_df["Overlap"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5, str(val), ha="center", va="bottom", fontsize=10)
ax.set_title("Top-50 overlap between anomaly detection methods")
ax.set_ylabel("Document count")
ax.set_ylim(0, 85)
ax.legend()
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(results_dir / "notebook_05_method_overlap.png", dpi=120, bbox_inches="tight")
plt.show()

overlap_df

The low overlap between methods reflects their fundamentally different detection strategies. IF and LOF share some common detections because both use the full structural feature space, but their objectives differ enough to produce different rankings. The small IF ∩ kNN overlap is expected: kNN with cosine distance selects lonely points while IF selects points that are collectively separated from the main distribution.

---

## Ensemble detector — rank averaging all three methods

The `EnsembleAnomalyDetector` combines the three methods via normalised rank averaging:

1. Each method produces a raw score array (lower = more anomalous).
2. Each score array is converted to normalised ranks in [0, 1].
3. The three rank arrays are averaged element-wise.
4. The final combined score retains the convention that lower = more anomalous.

This approach is designed to be robust: no single method can dominate the ranking regardless of score scale. However, when one method is much more accurate than the others, simple averaging can *dilute* the good method's signal with the weaker methods' noise.

In [ ]:
ensemble_detector = EnsembleAnomalyDetector(
    contamination_ratio=0.025,
    n_neighbors_lof=50,
    n_neighbors_knn=20,
    random_seed=42,
)
ens_mask, ens_scores = ensemble_detector.run_detection(feature_matrix)

# Show top-50 overlap with individual methods.
ens_set = top_k_ids(ens_scores, doc_ids)

ens_overlap_data = [
    {"Pair": "Ensemble ∩ IF", "Overlap": len(ens_set & if_set)},
    {"Pair": "Ensemble ∩ LOF", "Overlap": len(ens_set & lof_set)},
    {"Pair": "Ensemble ∩ kNN", "Overlap": len(ens_set & knn_set)},
]
ens_overlap_df = pd.DataFrame(ens_overlap_data)
ens_overlap_df.to_csv(results_dir / "notebook_05_ensemble_overlap.csv", index=False)

print("Ensemble top-50 overlap with individual methods:")
print(ens_overlap_df.to_string(index=False))
print()
print("Mean structural features of Ensemble top-50:")
print(f"  Bigram TTR      : {feature_matrix[np.argsort(ens_scores)[:50], bigram_col].mean():.3f}")
print(f"  Compression ratio: {feature_matrix[np.argsort(ens_scores)[:50], comp_col].mean():.3f}")

### Ensemble vs individual method performance comparison

The cell below builds a summary table showing what fraction of each method's top-50 are structurally anomalous (low bigram TTR, low compression ratio). This acts as a proxy for precision when ground truth is unavailable.

In [ ]:
# Threshold: bigram TTR < 0.70 and compression ratio < 0.55 considered structurally anomalous.
BIGRAM_TTR_THRESHOLD = 0.70
COMP_RATIO_THRESHOLD = 0.55


def structural_precision(indices: np.ndarray) -> float:
    """Share of selected documents that are below both structural thresholds."""
    bttr = feature_matrix[indices, bigram_col] < BIGRAM_TTR_THRESHOLD
    cr = feature_matrix[indices, comp_col] < COMP_RATIO_THRESHOLD
    return float((bttr & cr).mean())


perf_rows = []
for label, scores in [
    ("Isolation Forest", if_scores),
    ("LOF", lof_scores),
    ("kNN", knn_scores),
    ("Ensemble", ens_scores),
]:
    idx = np.argsort(scores)[:50]
    perf_rows.append(
        {
            "Method": label,
            "Structural precision (top-50)": round(structural_precision(idx), 3),
            "Mean bigram TTR (top-50)": round(float(feature_matrix[idx, bigram_col].mean()), 3),
            "Mean compression ratio (top-50)": round(float(feature_matrix[idx, comp_col].mean()), 3),
        }
    )

perf_df = pd.DataFrame(perf_rows)
perf_df.to_csv(results_dir / "notebook_05_method_performance_summary.csv", index=False)
perf_df

**Interpretation:**

The structural precision metric confirms that Isolation Forest selects the most homogeneously anomalous group (all top-50 documents are below the structural thresholds). The ensemble's precision depends on how much weight the weaker detectors (LOF, kNN) have — in this case, their false positives reduce the ensemble's precision below IF alone.

This is an important methodological insight: **ensemble methods outperform individual methods when all component detectors are reasonably accurate**. When one component is substantially weaker, rank-averaging can dilute rather than amplify the signal. Weighted ensembles or detector selection strategies are a natural extension.

---

## Final top-50 selection and anomaly export

Isolation Forest on structural features is used for the final submission because it produces the highest structural precision among the tested methods. The output follows the required format: exactly 50 rows with `anomaly` rank and `doc_id`.

In [ ]:
final_anomaly_df = create_anomaly_output(
    document_ids=doc_ids,
    anomaly_mask=if_mask,
    anomaly_scores=if_scores,
    expected_anomaly_count=50,
)

final_with_text = attach_original_text_by_doc_id(
    anomaly_table=final_anomaly_df,
    document_ids=doc_ids,
    raw_texts=raw_texts,
)

final_with_text.to_csv(results_dir / "notebook_05_anomalies_candidate.csv", index=False)
final_anomaly_df.to_csv(results_dir / "anomalies.csv", index=False)

print(f"Exported {len(final_anomaly_df)} anomaly IDs to anomalies.csv")
print()
final_with_text.head(10)

### Files produced by this notebook

| File | Description |
|---|---|
| `data/results/notebook_05_structural_feature_distributions.png` | Bigram TTR and compression ratio histograms |
| `data/results/notebook_05_if_score_distribution.png` | Isolation Forest score distribution with threshold marker |
| `data/results/notebook_05_isolation_forest_ranking.csv` | Full IF ranking for all 2 164 documents |
| `data/results/notebook_05_lof_ranking.csv` | Full LOF ranking for all 2 164 documents |
| `data/results/notebook_05_knn_ranking.csv` | Full kNN ranking for all 2 164 documents |
| `data/results/notebook_05_detector_top50_profile.csv` | Structural feature means and internal distance for each detector's top-50 |
| `data/results/notebook_05_method_overlap.csv` | Pairwise and union/intersection overlap between the three methods |
| `data/results/notebook_05_method_overlap.png` | Bar chart of top-50 overlap counts |
| `data/results/notebook_05_ensemble_overlap.csv` | Ensemble vs individual method top-50 overlap |
| `data/results/notebook_05_method_performance_summary.csv` | Structural precision comparison across all methods |
| `data/results/notebook_05_anomalies_candidate.csv` | Final top-50 anomalies with raw text snippets |
| `data/results/anomalies.csv` | **Assignment submission file** — 50 anomaly IDs in required format |